# 09. Dataset 품질검사와 시각화

생성된 image, mask, overlay, split summary를 확인합니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
    search_roots = [Path.cwd(), *Path.cwd().parents]
    search_patterns = ["synthetic_metal_utils.py", "*/synthetic_metal_utils.py", "*/*/synthetic_metal_utils.py"]
    for root in search_roots:
        for pattern in search_patterns:
            matches = list(root.glob(pattern))
            if matches:
                NOTEBOOK_DIR = matches[0].parent
                break
        if (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
            break

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "synthetic_metal_seg"

from synthetic_metal_utils import *
set_korean_font()
set_seed(7)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("DATA_ROOT:", DATA_ROOT)

## 필요하면 데이터셋 생성

08번을 아직 실행하지 않았다면 작은 기본 dataset을 생성합니다.

In [ ]:
if not (DATA_ROOT / "metadata" / "samples.csv").exists():
    create_synthetic_metal_dataset(DATA_ROOT, n_train=120, n_eval_per_split=24, overwrite=True)
create_preview_artifacts(DATA_ROOT)
summary = inspect_dataset(DATA_ROOT)
summary

## preview 이미지 열기

previews 폴더의 sample grid와 overlay grid를 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

preview_paths = [
    DATA_ROOT / "previews" / "sample_grid.png",
    DATA_ROOT / "previews" / "mask_overlay_grid.png",
    DATA_ROOT / "previews" / "domain_color_histograms.png",
    DATA_ROOT / "previews" / "split_summary.png",
]
fig, axes = plt.subplots(len(preview_paths), 1, figsize=(12, 12))
for ax, path in zip(axes, preview_paths):
    ax.imshow(Image.open(path))
    ax.set_title(path.name)
    ax.axis("off")
plt.tight_layout()

## sample 직접 확인

split별 image/mask/overlay를 직접 봅니다.

In [ ]:
rows = read_samples(DATA_ROOT, "val_unseen_light")[:6]
fig, axes = plt.subplots(3, len(rows), figsize=(10, 5))
for idx, row in enumerate(rows):
    image, mask = load_image_mask(DATA_ROOT, row)
    axes[0, idx].imshow(image); axes[0, idx].axis("off")
    axes[1, idx].imshow(mask, cmap="gray"); axes[1, idx].axis("off")
    axes[2, idx].imshow(make_overlay(image, mask)); axes[2, idx].axis("off")
plt.tight_layout()